In [1]:
from torchvision import datasets, utils, transforms
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
import torch.nn.functional as F
import pathlib
from PIL import Image

In [2]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EPOCHS = 30

# 데이터셋 만들기

In [3]:
class DataSetLoad(Dataset):
    def __init__(self, path, transform=None):
        self.path = path
        self.imgs = list(self.path.glob('*.jpg'))
        self.transform = transform
        if len(self.imgs) == 0:
            print('jpg 이미지가 존재하지 않습니다.')

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, index):
        one_img = self.imgs[index]
        img = Image.open(one_img).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = one_img.name.split('.')[0].lower()
        label_num = -1
        if label == 'cat':
            label_num = 0
        elif label == 'dog':
            label_num = 1
        else:
            raise ValueError(f"데이터의 클래스를 확인할 수 없습니다.{one_img.name}")
        
        return img, label_num

In [4]:
filepath = pathlib.Path('dogs-vs-cats/train')
transform = transforms.Compose([
    transforms.Resize([100,100]),
    transforms.ToTensor()
])
datasetload = DataSetLoad(filepath, transform=transform)
datasetload.__getitem__(1)

(tensor([[[0.1569, 0.1569, 0.1804,  ..., 0.4471, 0.7255, 0.7882],
          [0.1451, 0.1412, 0.1608,  ..., 0.4431, 0.7020, 0.7843],
          [0.1529, 0.1373, 0.1451,  ..., 0.4275, 0.7176, 0.8000],
          ...,
          [0.0824, 0.0784, 0.0863,  ..., 0.2941, 0.3098, 0.2549],
          [0.0941, 0.0824, 0.0863,  ..., 0.2510, 0.2157, 0.1804],
          [0.1098, 0.0863, 0.0941,  ..., 0.3098, 0.1882, 0.1765]],
 
         [[0.1765, 0.1725, 0.1922,  ..., 0.4588, 0.7176, 0.7765],
          [0.1647, 0.1529, 0.1725,  ..., 0.4549, 0.6902, 0.7608],
          [0.1725, 0.1490, 0.1569,  ..., 0.4392, 0.6980, 0.7647],
          ...,
          [0.0784, 0.0745, 0.0784,  ..., 0.2235, 0.2510, 0.2039],
          [0.0863, 0.0745, 0.0784,  ..., 0.1843, 0.1569, 0.1333],
          [0.1020, 0.0784, 0.0863,  ..., 0.2431, 0.1294, 0.1333]],
 
         [[0.1647, 0.1804, 0.2196,  ..., 0.4667, 0.6431, 0.6510],
          [0.1529, 0.1647, 0.1961,  ..., 0.4745, 0.6275, 0.6510],
          [0.1608, 0.1608, 0.1804,  ...,

In [5]:
ds_n = len(datasetload)
ds_n

25000

In [6]:
tr_ds_n, tt_ds_n = map(int, [ds_n*0.7, ds_n*0.3])
val_ds_n = int(tr_ds_n*0.2)
tr_ds_n = int(tr_ds_n*0.8)
tr_ds_n, tt_ds_n, val_ds_n, sum([tr_ds_n, tt_ds_n, val_ds_n])

(14000, 7500, 3500, 25000)

In [7]:
tr_ds, tt_ds, val_ds = random_split(datasetload, [tr_ds_n, tt_ds_n, val_ds_n])
len(tr_ds), len(tt_ds), len(val_ds)

(14000, 7500, 3500)

In [8]:
tr_ds_loader = DataLoader(tr_ds, batch_size=32, shuffle=True)
tt_ds_loader = DataLoader(tt_ds, batch_size=32, shuffle=True)
val_ds_loader = DataLoader(val_ds, batch_size=32, shuffle=True)

# DNN

In [ ]:
class DNN(nn.Module):
    def __init__(self):
        super().__init__()
        # self.fc1 = nn.Linear(3*180*180, 4096)
        self.fc1 = nn.Linear(3*100*100, 4096)
        self.fc2 = nn.Linear(4096, 1024)
        self.fc3 = nn.Linear(1024, 256)
        self.fc4 = nn.Linear(256, 2)

    def forward(self, x):
        x = x.view(-1, 3*100*100) # 어차피 지정한 데이터 크기로 들어갈 거라 고정으로 지정
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x # 활성함수까지 바로 적용

In [ ]:
dnn = DNN().to(DEVICE)
opt = optim.Adam(dnn.parameters()) # 그냥 디폴트값으로 진행해보는 걸로

In [ ]:
from tqdm import tqdm
def train(m, data, opt):
    m.train()
    for x,y in tqdm(data):
        dat, target = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        output = m(dat)
        loss = F.cross_entropy(output, target)
        loss.backward()
        opt.step()

In [ ]:
def evaluate(m, data):
    m.eval()
    t_loss, correct = 0, 0
    with torch.no_grad():
        for d, t in data:
            d, t = d.to(DEVICE), t.to(DEVICE)
            output = m(d)
            t_loss += F.cross_entropy(output, t, reduction='sum').item()
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(t.view_as(pred)).sum().item()
    t_loss /= len(data.dataset)
    t_acc = correct/len(data.dataset) * 100
    return t_loss, t_acc

In [ ]:
# for i in range(1, EPOCHS+1): # 이미지 180일때, 너무 느리고 정확도도 이상함
#     train(dnn, tr_ds_loader, opt)
#     test_loss, test_acc = evaluate(dnn, tt_ds_loader)
#     print(f'epoch:{i}\tloss:{test_loss}\taccuracy:{test_acc}')

epoch:1	loss:0.6707068517049154	accuracy:58.653333333333336
epoch:2	loss:0.6855409764607747	accuracy:55.09333333333334
epoch:3	loss:0.6726839739481608	accuracy:56.42666666666667
epoch:4	loss:0.666453675587972	accuracy:60.29333333333333
epoch:5	loss:0.6931386217753093	accuracy:49.76
epoch:6	loss:0.6932132659912109	accuracy:50.32
epoch:7	loss:0.6858161930084229	accuracy:54.81333333333333


KeyboardInterrupt: 

In [ ]:
for i in range(1, EPOCHS+1): # 이미지 100
    train(dnn, tr_ds_loader, opt)
    test_loss, test_acc = evaluate(dnn, tt_ds_loader)
    print(f'epoch:{i}\tloss:{test_loss}\taccuracy:{test_acc}')

100%|██████████| 438/438 [00:36<00:00, 12.05it/s]


epoch:1	loss:0.6842801497777303	accuracy:57.986666666666665


100%|██████████| 438/438 [00:33<00:00, 13.17it/s]


epoch:2	loss:0.6589044006983439	accuracy:60.84


100%|██████████| 438/438 [00:32<00:00, 13.37it/s]


...

epoch:10	loss:0.6365754517873128	accuracy:62.906666666666666


100%|██████████| 438/438 [00:42<00:00, 10.23it/s]


epoch:11	loss:0.6636570262908935	accuracy:62.32


  5%|▌         | 24/438 [00:02<00:43,  9.46it/s]


KeyboardInterrupt: 

In [33]:
EPOCHS = 10

In [40]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, 1, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1, 1)
        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128*12*12,4096)
        self.fc2 = nn.Linear(4096,1024)
        self.fc3 = nn.Linear(1024,128)
        self.fc4 = nn.Linear(128,2)

    def forward(self, x):
        x = F.max_pool2d(self.conv1(x), 2) # 32 50 50 
        x = F.max_pool2d(self.conv2(x), 2) # 64 25 25
        x = F.max_pool2d(self.conv3(x), 2) # 128 12 12
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = F.relu(self.fc2(x))
        x = F.dropout(x, training=self.training)
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x

In [41]:
for i, j in tr_ds_loader:
    print(i.shape) # [32, 3, 100, 100]
    break

torch.Size([32, 3, 100, 100])


In [42]:
cnn = CNN().to(DEVICE)
opt = optim.Adam(cnn.parameters())

In [43]:
from tqdm import tqdm
def train(m, data, opt):
    m.train()
    for x,y in tqdm(data):
        dat, target = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        output = m(dat)
        loss = F.cross_entropy(output, target)
        loss.backward()
        opt.step()
def evaluate(m, data):
    m.eval()
    t_loss, correct = 0, 0
    with torch.no_grad():
        for d, t in data:
            d, t = d.to(DEVICE), t.to(DEVICE)
            output = m(d)
            t_loss += F.cross_entropy(output, t, reduction='sum').item()
            pred = output.max(1, keepdim=True)[1]
            correct += pred.eq(t.view_as(pred)).sum().item()
    t_loss /= len(data.dataset)
    t_acc = correct/len(data.dataset) * 100
    return t_loss, t_acc

In [44]:
for i in range(1, EPOCHS+1): # 이미지 100
    train(cnn, tr_ds_loader, opt)
    test_loss, test_acc = evaluate(cnn, tt_ds_loader)
    print(f'epoch:{i}\tloss:{test_loss}\taccuracy:{test_acc}')

100%|██████████| 438/438 [00:29<00:00, 14.74it/s]


epoch:1	loss:0.6114243267695109	accuracy:67.56


100%|██████████| 438/438 [00:30<00:00, 14.30it/s]


epoch:2	loss:0.530979946422577	accuracy:73.72


100%|██████████| 438/438 [00:30<00:00, 14.54it/s]


...

epoch:8	loss:0.933520412349701	accuracy:79.22666666666667


100%|██████████| 438/438 [00:40<00:00, 10.89it/s]


epoch:9	loss:0.8708051178296407	accuracy:79.12


100%|██████████| 438/438 [00:37<00:00, 11.83it/s]


epoch:10	loss:0.9868252070109049	accuracy:77.98666666666666
